In [1]:
# ABOUTME: Interactive notebook showing how neural network representations evolve through layers
# ABOUTME: Uses red/blue tinted pet images and jscatter linked views to visualize layer-by-layer clustering

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torchvision
from torchvision import models, transforms
from torchvision.models.feature_extraction import create_feature_extractor
from sklearn.decomposition import PCA
from PIL import Image
from ipywidgets import interact, widgets
from torchinfo import summary
import jscatter
import base64
from io import BytesIO

%matplotlib widget


def pil_to_data_uri(img, size=128, fmt='JPEG'):
    """Convert a PIL image to a base64 data URI for jscatter tooltip preview."""
    thumb = img.copy()
    thumb.thumbnail((size, size))
    buf = BytesIO()
    thumb.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    mime = 'image/jpeg' if fmt == 'JPEG' else 'image/png'
    return f"data:{mime};base64,{b64}"

# How Representations Evolve: From Colors to Concepts

A neural network has many layers, stacked on top of each other. Each layer transforms the data a little further — but what does each one *see*?

**The experiment:** We'll take photos of dogs and cats, tint some **red** and some **blue**, then feed them through a pretrained ResNet-18. At each layer, we'll extract the representations and plot them.

**The prediction:**
- **Early layers** should cluster images by **color** (red vs blue) — because early layers detect low-level features like edges, textures, and colors
- **Late layers** should cluster images by **animal** (dog vs cat) — because late layers detect high-level concepts like shapes, parts, and objects

Let's see if the network agrees.

In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

N_PER_GROUP = 10        # Images per group (10 red dogs, 10 blue dogs, etc.)
TINT_INTENSITY = 0.35   # How strong the color tint is (0 = none, 1 = solid color)
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "./data"
THUMBNAIL_SIZE = 128

# Layers to extract features from
LAYER_NAMES = ['layer1', 'layer2', 'layer4', 'avgpool']

print(f"Device: {DEVICE}")
print(f"Images per group: {N_PER_GROUP}")
print(f"Total images: {N_PER_GROUP * 4} (red dogs, blue dogs, red cats, blue cats)")
print(f"Tint intensity: {TINT_INTENSITY}")
print(f"Layers to extract: {LAYER_NAMES}")

In [3]:
# ============================================================================
# LOAD DATASET AND APPLY COLOR TINTING
# ============================================================================

np.random.seed(SEED)
torch.manual_seed(SEED)


def apply_color_tint(image, color, intensity=0.35):
    """Apply a red or blue color overlay to an image."""
    tint_rgb = (255, 50, 50) if color == 'red' else (50, 50, 255)
    overlay = Image.new('RGB', image.size, tint_rgb)
    return Image.blend(image, overlay, intensity)


# Load Oxford Pets
dataset = torchvision.datasets.OxfordIIITPet(
    root=DATA_DIR, split='trainval',
    target_types='binary-category',
    download=True
)

# Separate by class (0 = Cat, 1 = Dog)
cat_indices = [i for i, (_, label) in enumerate(dataset) if label == 0]
dog_indices = [i for i, (_, label) in enumerate(dataset) if label == 1]

# Sample and tint
sampled_cat_idx = np.random.choice(cat_indices, N_PER_GROUP * 2, replace=False)
sampled_dog_idx = np.random.choice(dog_indices, N_PER_GROUP * 2, replace=False)

images = []
labels = []           # "Dog" or "Cat"
dominant_colors = []  # "Red" or "Blue"
groups = []           # "Red Dog", "Blue Dog", etc.

for i, idx in enumerate(sampled_dog_idx):
    img, _ = dataset[idx]
    color = 'red' if i < N_PER_GROUP else 'blue'
    tinted = apply_color_tint(img, color, TINT_INTENSITY)
    images.append(tinted)
    labels.append('Dog')
    dominant_colors.append('Red' if color == 'red' else 'Blue')
    groups.append(f'{"Red" if color == "red" else "Blue"} Dog')

for i, idx in enumerate(sampled_cat_idx):
    img, _ = dataset[idx]
    color = 'red' if i < N_PER_GROUP else 'blue'
    tinted = apply_color_tint(img, color, TINT_INTENSITY)
    images.append(tinted)
    labels.append('Cat')
    dominant_colors.append('Red' if color == 'red' else 'Blue')
    groups.append(f'{"Red" if color == "red" else "Blue"} Cat')

print(f"Created {len(images)} tinted images:")
for g in ['Red Dog', 'Blue Dog', 'Red Cat', 'Blue Cat']:
    print(f"  {g}: {groups.count(g)}")

In [4]:
# ============================================================================
# DISPLAY IMAGE GRID
# ============================================================================

fig, axes = plt.subplots(4, N_PER_GROUP, figsize=(2 * N_PER_GROUP, 8))

group_order = ['Red Dog', 'Blue Dog', 'Red Cat', 'Blue Cat']
group_colors = {'Red Dog': '#cc3333', 'Blue Dog': '#3333cc',
                'Red Cat': '#cc3333', 'Blue Cat': '#3333cc'}

for row, group_name in enumerate(group_order):
    group_imgs = [images[i] for i in range(len(images)) if groups[i] == group_name]
    for col, img in enumerate(group_imgs[:N_PER_GROUP]):
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(group_name, fontsize=12, fontweight='bold',
                             rotation=0, labelpad=80, va='center',
                             color=group_colors[group_name])

fig.suptitle('Our Tinted Dataset: 4 Groups', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Looking Through the Network's Eyes

ResNet-18 is a deep neural network with many layers stacked on top of each other. Each layer transforms the image a little further, building increasingly abstract representations.

Let's look at its actual structure, then visualize which layers we'll extract features from.

In [5]:
# ============================================================================
# RESNET-18 ARCHITECTURE SUMMARY
# ============================================================================

# Load model to inspect its structure
resnet_preview = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# torchinfo gives us a clean summary of every layer, its output shape, and parameter count
summary(resnet_preview, input_size=(1, 3, 224, 224), depth=2,
        col_names=["input_size", "output_size", "num_params"])

In [6]:
# ============================================================================
# VISUAL ARCHITECTURE DIAGRAM
# ============================================================================

fig_arch, ax_arch = plt.subplots(figsize=(14, 5))
ax_arch.set_xlim(-0.5, 13)
ax_arch.set_ylim(-1, 3.5)
ax_arch.axis('off')

# Define layers with their properties
layers = [
    {'name': 'Input\n3×224×224', 'x': 0, 'color': '#95a5a6', 'width': 1.2,
     'sampled': False, 'channels': 3},
    {'name': 'conv1 + bn + relu\n+ maxpool', 'x': 1.8, 'color': '#bdc3c7', 'width': 1.5,
     'sampled': False, 'channels': 64},
    {'name': 'layer1\n64×56×56', 'x': 3.8, 'color': '#e74c3c', 'width': 1.3,
     'sampled': True, 'channels': 64},
    {'name': 'layer2\n128×28×28', 'x': 5.6, 'color': '#e67e22', 'width': 1.3,
     'sampled': True, 'channels': 128},
    {'name': 'layer3\n256×14×14', 'x': 7.4, 'color': '#bdc3c7', 'width': 1.3,
     'sampled': False, 'channels': 256},
    {'name': 'layer4\n512×7×7', 'x': 9.2, 'color': '#2ecc71', 'width': 1.3,
     'sampled': True, 'channels': 512},
    {'name': 'avgpool\n512×1×1', 'x': 11.0, 'color': '#3498db', 'width': 1.3,
     'sampled': True, 'channels': 512},
]

# Draw layers as blocks with height proportional to channel count (log scale)
for layer in layers:
    height = 0.5 + np.log2(layer['channels']) * 0.2
    y = 1.0 - height / 2

    edgecolor = 'black' if layer['sampled'] else '#7f8c8d'
    linewidth = 3 if layer['sampled'] else 1

    rect = mpatches.FancyBboxPatch(
        (layer['x'], y), layer['width'], height,
        boxstyle="round,pad=0.05",
        facecolor=layer['color'],
        edgecolor=edgecolor,
        linewidth=linewidth,
        alpha=0.85,
    )
    ax_arch.add_patch(rect)
    ax_arch.text(layer['x'] + layer['width'] / 2, y + height / 2,
                 layer['name'], ha='center', va='center',
                 fontsize=8, fontweight='bold' if layer['sampled'] else 'normal',
                 color='white' if layer['sampled'] else 'black')

    # "We sample here" markers
    if layer['sampled']:
        ax_arch.annotate(
            'WE SAMPLE\nHERE',
            xy=(layer['x'] + layer['width'] / 2, y - 0.05),
            xytext=(layer['x'] + layer['width'] / 2, y - 0.7),
            ha='center', va='top', fontsize=7, fontweight='bold',
            color=layer['color'],
            arrowprops=dict(arrowstyle='->', color=layer['color'], lw=2),
        )

# Draw arrows between layers
for i in range(len(layers) - 1):
    x_start = layers[i]['x'] + layers[i]['width']
    x_end = layers[i + 1]['x']
    ax_arch.annotate(
        '', xy=(x_end, 1.0), xytext=(x_start, 1.0),
        arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5),
    )

ax_arch.set_title('ResNet-18 Architecture — Layers We Extract Features From',
                   fontsize=14, fontweight='bold', pad=15)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#e74c3c', edgecolor='black', linewidth=2,
                   label='Sampled layers (features extracted)'),
    mpatches.Patch(facecolor='#bdc3c7', edgecolor='#7f8c8d', linewidth=1,
                   label='Other layers (not sampled)'),
]
ax_arch.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

The colored blocks are the layers we'll extract features from. Notice how the spatial dimensions **shrink** (224→56→28→14→7→1) while the number of channels **grows** (3→64→128→256→512). The network trades spatial detail for semantic richness.

| Layer | Channels | Spatial Size | What it sees |
|-------|----------|-------------|--------------|
| **layer1** | 64 | 56×56 | Edges, colors, textures |
| **layer2** | 128 | 28×28 | Corners, patterns, gradients |
| **layer4** | 512 | 7×7 | Object parts, shapes |
| **avgpool** | 512 | 1×1 | Pure concept summary |

We'll extract features from each, then project to 2D with PCA.

In [7]:
# ============================================================================
# EXTRACT FEATURES FROM ALL LAYERS
# ============================================================================

# Load ResNet-18 with intermediate feature extraction
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.eval()
resnet = resnet.to(DEVICE)

return_nodes = {name: name for name in LAYER_NAMES}
feature_extractor = create_feature_extractor(resnet, return_nodes=return_nodes)

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Extract features for all images at all layers
layer_features = {name: [] for name in LAYER_NAMES}

print("Extracting features from all layers...")
with torch.no_grad():
    for img in images:
        tensor = preprocess(img).unsqueeze(0).to(DEVICE)
        features = feature_extractor(tensor)

        for name in LAYER_NAMES:
            feat = features[name]
            # Global average pooling to collapse spatial dimensions
            if feat.dim() == 4:
                feat = feat.mean(dim=[2, 3])  # [B, C, H, W] → [B, C]
            feat = feat.squeeze().cpu().numpy()
            layer_features[name].append(feat)

# Convert to numpy arrays
for name in LAYER_NAMES:
    layer_features[name] = np.array(layer_features[name])
    print(f"  {name:10s}: {layer_features[name].shape}")

In [8]:
# ============================================================================
# PCA PROJECT EACH LAYER TO 2D + BUILD DATAFRAME
# ============================================================================

# Build a unified DataFrame with PCA coordinates for each layer
df_data = {
    'label': pd.Categorical(labels),
    'dominant_color': pd.Categorical(dominant_colors),
    'group': pd.Categorical(groups),
}

for name in LAYER_NAMES:
    pca = PCA(n_components=2)
    coords = pca.fit_transform(layer_features[name])
    df_data[f'{name}_x'] = coords[:, 0]
    df_data[f'{name}_y'] = coords[:, 1]
    var_explained = sum(pca.explained_variance_ratio_) * 100
    print(f"  {name:10s}: PCA variance explained = {var_explained:.1f}%")

# Generate data URIs for jscatter tooltips
df_data['thumbnail'] = [pil_to_data_uri(img, size=THUMBNAIL_SIZE) for img in images]

df = pd.DataFrame(df_data)

print(f"\nDataFrame shape: {df.shape}")
df.head()

## Layer 1: The Network Sees... Colors!

At the earliest layer, the network has only processed raw pixel patterns — edges, colors, textures. Let's color the points by their **dominant color** (red vs blue). Do the red images cluster together? Do the blue ones?

In [9]:
# ============================================================================
# LAYER 1: COLOR-BASED CLUSTERING
# ============================================================================

scatter_early = jscatter.Scatter(
    data=df, x='layer1_x', y='layer1_y',
    height=450,
)
scatter_early.color(by='dominant_color', map={'Red': '#e74c3c', 'Blue': '#3498db'})
scatter_early.size(8)
scatter_early.tooltip(
    enable=True,
    properties=['label', 'dominant_color', 'group'],
    preview='thumbnail',
    preview_type='image',
    preview_image_size='contain',
)
scatter_early.legend(True)
scatter_early.show()

## Final Layer: The Network Sees... Animals!

By the final layer, the network has built up abstract concepts. Colors and textures have been abstracted away. Now let's color by **animal type** (dog vs cat). Does the clustering change?

In [10]:
# ============================================================================
# FINAL LAYER: SEMANTIC CLUSTERING
# ============================================================================

scatter_late = jscatter.Scatter(
    data=df, x='avgpool_x', y='avgpool_y',
    height=450,
)
scatter_late.color(by='label', map={'Dog': '#e74c3c', 'Cat': '#3498db'})
scatter_late.size(8)
scatter_late.tooltip(
    enable=True,
    properties=['label', 'dominant_color', 'group'],
    preview='thumbnail',
    preview_type='image',
    preview_image_size='contain',
)
scatter_late.legend(True)
scatter_late.show()

## The Full Journey — All Layers at Once

This is the headline visualization. Each plot shows the **same images** at a different layer of the network. The plots are **linked**: hover over a point in one plot and it highlights in all the others. Select a group in one plot and watch where they end up in the others.

Colors represent the 4 groups: Red Dog, Blue Dog, Red Cat, Blue Cat.

In [11]:
# ============================================================================
# LINKED MULTI-LAYER VIEW
# ============================================================================

group_colormap = {
    'Red Dog': '#e74c3c',
    'Blue Dog': '#3498db',
    'Red Cat': '#e67e22',
    'Blue Cat': '#2ecc71',
}

scatters = []
for name in LAYER_NAMES:
    s = jscatter.Scatter(data=df, x=f'{name}_x', y=f'{name}_y')
    s.color(by='group', map=group_colormap)
    s.size(8)
    s.tooltip(
        enable=True,
        properties=['label', 'dominant_color', 'group'],
        preview='thumbnail',
        preview_type='image',
        preview_image_size='contain',
    )
    s.legend(True)
    scatters.append(s)

jscatter.link(scatters, rows=2)

### Try This

1. In the **layer1** plot, lasso-select the cluster of red images. Watch where those same points appear in **avgpool** — are they still together?
2. In the **avgpool** plot, lasso-select the dog cluster. Now look at **layer1** — the dogs were scattered by color!
3. Notice how the transition from color-based to animal-based clustering is gradual across layers.

## Switch Perspectives

Use the dropdown below to re-color a single plot by different attributes and see how the clustering story changes.

In [12]:
# ============================================================================
# INTERACTIVE LAYER + COLOR-BY EXPLORER
# ============================================================================

color_maps = {
    'label': {'Dog': '#e74c3c', 'Cat': '#3498db'},
    'dominant_color': {'Red': '#e74c3c', 'Blue': '#3498db'},
    'group': group_colormap,
}

scatter_explore = jscatter.Scatter(
    data=df, x='avgpool_x', y='avgpool_y',
    height=450,
)
scatter_explore.color(by='group', map=group_colormap)
scatter_explore.size(8)
scatter_explore.tooltip(
    enable=True,
    properties=['label', 'dominant_color', 'group'],
    preview='thumbnail',
    preview_type='image',
    preview_image_size='contain',
)
scatter_explore.legend(True)


@interact(
    layer=widgets.Dropdown(
        options=LAYER_NAMES,
        value='avgpool',
        description='Layer:',
    ),
    color_by=widgets.Dropdown(
        options=['label', 'dominant_color', 'group'],
        value='group',
        description='Color by:',
    ),
)
def update_scatter(layer, color_by):
    scatter_explore.xy(f'{layer}_x', f'{layer}_y')
    scatter_explore.color(by=color_by, map=color_maps[color_by])


scatter_explore.show()

## Why Does This Happen?

Each layer of a convolutional neural network builds on the previous one:

- **Layer 1** has tiny filters (3×3 pixels) that detect basic patterns: edges, gradients, **color patches**. At this level, a red dog and a red cat both light up the same "red detector" filters. So they look similar.

- **Layer 2** combines layer 1's patterns into slightly larger patterns: corners, simple textures, color gradients.

- **Layer 4** combines everything into high-level detectors: "furry ear shape", "whisker pattern", "snout shape". These respond to animal identity, not color.

- **avgpool** squeezes the spatial information away, leaving a pure concept vector: "this is a dog" or "this is a cat".

**The key insight:** Deep networks learn a hierarchy of representations. Each layer adds abstraction. Early layers see pixels; late layers see meaning. This is why deep networks need to be *deep* — you can't jump from pixels to concepts in one step.

## Going Further

- **Notebook 03** showed how embeddings work at the final layer
- **Notebook 05** will show how to use embeddings for similarity search
- **Notebook 06** will show how text and images can share the same embedding space
- The `PreciousZoo/DINOv2/` directory contains research on a more modern architecture (Vision Transformers)